# Movie Night Recommender — Agentic RAG Demo

A simple **agentic workflow** built with **LangGraph** + **RAG** (Retrieval-Augmented Generation).

**The idea:** you describe a mood or ask about a film, a *router* decides how to handle it, and the answer is grounded in a small movie knowledge base.

```
            +-------------------+
  query --> |      router       |
            +---------+---------+
                      |
        +-------------+--------------+----------------+
        v             v              v
   recommend_node   rag_node      trivia_node
  (retrieve+LLM)  (retrieve+LLM)  (curated list)
```

**Why this is a good teaching example:**
- `recommend_node` uses retrieval to match your *mood* to real movies.
- `rag_node` answers factual questions grounded in the movie blurbs.
- A distance threshold lets the agent say **"I don't know"** for off-topic requests.

## 1. Setup

These packages are already installed in the course environment. If you are running somewhere new (e.g. Colab), uncomment the pip line below.

We use a small, fast, instruction-tuned model (`flan-t5-small`, ~300 MB) so this runs in minutes with **no API key**.

In [ ]:
# !pip install -q faiss-cpu sentence-transformers transformers langgraph

import random
import numpy as np
import faiss
import torch
from typing import TypedDict
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langgraph.graph import StateGraph, END

print("Imports OK")

## 2. Knowledge base + retriever (the "R" in RAG)

We embed a small movie catalog (title, genre, and a one-line plot) and store it in a FAISS vector index. `retrieve()` returns the closest movies **with their distances**.

In [ ]:
movies = [
    "Inception (Sci-Fi/Thriller): A thief who steals secrets from dreams is hired to plant an idea instead.",
    "The Grand Budapest Hotel (Comedy): A quirky concierge and his protege get tangled up in the theft of a priceless painting.",
    "Finding Nemo (Animation/Family): A worried clownfish crosses the ocean to find his lost son.",
    "The Dark Knight (Action/Crime): Batman faces the Joker, a mastermind who wants to plunge Gotham into chaos.",
    "Pride and Prejudice (Romance/Drama): Elizabeth Bennet navigates love and pride with the reserved Mr. Darcy.",
    "Interstellar (Sci-Fi/Drama): Astronauts travel through a wormhole to find a new home for humanity.",
    "Spirited Away (Animation/Fantasy): A young girl enters a spirit world and must work to free her parents.",
    "Get Out (Horror/Thriller): A man uncovers a disturbing secret while visiting his girlfriend's family.",
    "La La Land (Romance/Musical): An aspiring actress and a jazz musician chase their dreams in Los Angeles.",
    "The Martian (Sci-Fi/Adventure): An astronaut stranded on Mars uses science to survive and signal for rescue.",
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embed_model.encode(movies)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.array(kb_embeddings))
print(f"Indexed {index.ntotal} movies.")

In [ ]:
# Lower distance = more similar. Tune this threshold to control when the agent says "I don't know".
DISTANCE_THRESHOLD = 1.4

def retrieve(query, top_k=3):
    query_emb = embed_model.encode([query])
    distances, indices = index.search(np.array(query_emb), top_k)
    return [(movies[i], float(d)) for d, i in zip(distances[0], indices[0])]

# Quick sanity check
retrieve("something with space and science")

## 3. The generator (the "G" in RAG)

A small instruction-tuned model turns the retrieved movies into a friendly recommendation. The first run downloads the model (~300 MB).

In [ ]:
# flan-t5-small is a small instruction-tuned seq2seq model (~300 MB on first run).
# We load it directly because the transformers 5.x pipeline no longer exposes a
# "text2text-generation" task.
MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def call_llm(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output_ids = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print(call_llm("Recommend a feel-good movie in one sentence."))

## 4. Define the state and the nodes

- `recommend_node` — retrieves movies that match your mood and explains the pick
- `rag_node` — answers a factual question grounded in a movie blurb (with an "I don't know" fallback)
- `trivia_node` — shares a fun fact from a curated list

In [ ]:
class MovieState(TypedDict):
    query: str
    answer: str
    selected_node: str

TRIVIA = [
    "The spinning top at the end of 'Inception' is one of the most debated final shots in film history.",
    "'Spirited Away' was the first (and so far only) hand-drawn, non-English film to win the Best Animated Feature Oscar.",
    "The potatoes grown in 'The Martian' made the movie a favorite in real NASA outreach talks.",
]

def router_node(state: MovieState):
    q = state["query"].lower()
    if any(w in q for w in ["recommend", "suggest", "mood", "watch", "feel", "something"]):
        state["selected_node"] = "recommend_node"
    elif "trivia" in q or "fun fact" in q or "fun" in q:
        state["selected_node"] = "trivia_node"
    else:
        state["selected_node"] = "rag_node"
    return state

def recommend_node(state: MovieState):
    print("Recommendation node activated \U0001F37F")
    results = retrieve(state["query"], top_k=3)
    if results[0][1] > DISTANCE_THRESHOLD:
        state["answer"] = "I couldn't find a good match in my catalog. Try describing a genre or mood!"
        return state
    picks = [text.split(" (")[0] for text, _ in results]
    context = "\n".join(text for text, _ in results)
    prompt = (
        "You are a movie night host. Based on the movies below, suggest what to watch "
        "and give a one-sentence reason.\n"
        f"Movies:\n{context}\n\nRequest: {state['query']}\nRecommendation:"
    )
    state["answer"] = f"Top picks: {', '.join(picks)}.\n{call_llm(prompt)}"
    return state

def rag_node(state: MovieState):
    print("RAG node activated \U0001F3AC")
    results = retrieve(state["query"], top_k=2)
    if results[0][1] > DISTANCE_THRESHOLD:
        state["answer"] = "That movie isn't in my catalog. Ask me about one of the films I know!"
        return state
    context = "\n".join(text for text, _ in results)
    prompt = (
        "You are a film expert. Use the context to answer the question.\n"
        f"Context:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    )
    state["answer"] = call_llm(prompt)
    return state

def trivia_node(state: MovieState):
    print("Trivia node activated \U0001F3AC\u2728")
    state["answer"] = random.choice(TRIVIA)
    return state

## 5. Build the LangGraph workflow

The router is the entry point. Its output (`selected_node`) decides which branch runs via conditional edges.

In [ ]:
workflow = StateGraph(MovieState)
workflow.add_node("router", router_node)
workflow.add_node("recommend_node", recommend_node)
workflow.add_node("rag_node", rag_node)
workflow.add_node("trivia_node", trivia_node)

workflow.set_entry_point("router")
workflow.add_conditional_edges(
    "router",
    lambda s: s["selected_node"],
    {
        "recommend_node": "recommend_node",
        "rag_node": "rag_node",
        "trivia_node": "trivia_node",
    },
)
workflow.add_edge("recommend_node", END)
workflow.add_edge("rag_node", END)
workflow.add_edge("trivia_node", END)

app = workflow.compile()
print("Graph compiled \u2705")

In [ ]:
# Dependency-free diagram of the graph (paste into any Mermaid viewer to visualize).
print(app.get_graph().draw_mermaid())

## 6. Test it

Watch which node activates for each query. The last question is intentionally off-topic to trigger the fallback.

In [ ]:
queries = [
    "Recommend a sci-fi movie for tonight.",
    "What is Inception about?",
    "I want something romantic to watch.",
    "Give me some movie trivia!",
    "Which movie is set on Mars?",
    "What's a good pizza recipe?",
]

for q in queries:
    result = app.invoke({"query": q, "answer": "", "selected_node": ""})
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print("-" * 60)